# Example Quantitative Analysis

This notebook demonstrates basic usage of the quant trading framework.

In [ ]:
# Import required libraries
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import DataLoader
from src.strategies import SimpleMovingAverageCrossover
from src.portfolio import Portfolio

%matplotlib inline
sns.set_style('darkgrid')

## 1. Load Data

Fetch historical price data for analysis.

In [ ]:
# Initialize data loader
loader = DataLoader(data_source="yahoo")

# Fetch data
symbol = "AAPL"
data = loader.fetch_data(
    symbol=symbol,
    start_date="2023-01-01",
    end_date="2024-01-01"
)

print(f"Loaded {len(data)} rows of data for {symbol}")
data.head()

## 2. Visualize Price Data

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(data.index, data['close'], label='Close Price', linewidth=2)
plt.title(f'{symbol} Price History')
plt.xlabel('Date')
plt.ylabel('Price ($)')
plt.legend()
plt.grid(True)
plt.show()

## 3. Apply Trading Strategy

Test a Simple Moving Average Crossover strategy.

In [ ]:
# Initialize strategy
strategy = SimpleMovingAverageCrossover(
    short_window=20,
    long_window=50
)

# Generate signals
signals = strategy.generate_signals(data)
signals[['close', 'short_ma', 'long_ma', 'signal']].tail(10)

## 4. Visualize Strategy Signals

In [ ]:
plt.figure(figsize=(14, 8))

# Plot price and moving averages
plt.subplot(2, 1, 1)
plt.plot(signals.index, signals['close'], label='Close Price', alpha=0.7)
plt.plot(signals.index, signals['short_ma'], label=f'{strategy.short_window}-day MA', alpha=0.8)
plt.plot(signals.index, signals['long_ma'], label=f'{strategy.long_window}-day MA', alpha=0.8)
plt.title(f'{symbol} with Moving Averages')
plt.ylabel('Price ($)')
plt.legend()
plt.grid(True)

# Plot signals
plt.subplot(2, 1, 2)
plt.plot(signals.index, signals['signal'], label='Signal', color='purple', linewidth=2)
plt.axhline(y=0, color='black', linestyle='--', alpha=0.3)
plt.title('Trading Signals')
plt.xlabel('Date')
plt.ylabel('Signal')
plt.yticks([-1, 0, 1], ['Sell', 'Hold', 'Buy'])
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## 5. Calculate Basic Performance Metrics

In [ ]:
# Calculate returns
signals['returns'] = signals['close'].pct_change()
signals['strategy_returns'] = signals['signal'].shift(1) * signals['returns']

# Cumulative returns
signals['cumulative_returns'] = (1 + signals['returns']).cumprod()
signals['cumulative_strategy_returns'] = (1 + signals['strategy_returns']).cumprod()

# Plot cumulative returns
plt.figure(figsize=(14, 6))
plt.plot(signals.index, signals['cumulative_returns'], label='Buy & Hold', linewidth=2)
plt.plot(signals.index, signals['cumulative_strategy_returns'], label='Strategy', linewidth=2)
plt.title('Cumulative Returns Comparison')
plt.xlabel('Date')
plt.ylabel('Cumulative Returns')
plt.legend()
plt.grid(True)
plt.show()

# Performance summary
total_return = (signals['cumulative_returns'].iloc[-1] - 1) * 100
strategy_return = (signals['cumulative_strategy_returns'].iloc[-1] - 1) * 100

print(f"\nPerformance Summary:")
print(f"Buy & Hold Return: {total_return:.2f}%")
print(f"Strategy Return: {strategy_return:.2f}%")
print(f"Outperformance: {strategy_return - total_return:.2f}%")

## 6. Next Steps

- Try different strategy parameters
- Test on different symbols
- Implement risk management rules
- Calculate additional metrics (Sharpe ratio, max drawdown, etc.)
- Backtest on longer time periods